# WP33 — Formal Domain Adaptation: Lean Theorem Proving (v0.3: The Evolving Thinker)
## LeanTool · CurriculumAgent.generate_theorem · CoderAgent.prove

---

This notebook demonstrates **WP33: Formal Theorem Proving**, the third task of the
**v0.3 "Evolving Thinker"** phase. The system transitions from Python code generation
to generating *formal mathematical proofs* in **Lean 4**, verified by the Lean compiler.

### What WP33 Introduces

| Component | Role |
|-----------|------|
| **LeanTool** | Real subprocess integration with `lean`/`lake`; graceful mock fallback when Lean is not installed |
| **LeanTool._parse_lean_error()** | Parses JSON and plain-text Lean 4 diagnostics |
| **LeanTool.start_proof / apply_tactic** | Interactive proof-state management |
| **CurriculumAgent.generate_theorem()** | Returns `(theorem_str, difficulty)` at easy/medium/hard level |
| **CoderAgent.prove()** | Iterative retry loop: generate proof → verify with LeanTool → incorporate error feedback |

### Architecture

```
CurriculumAgent ──generate_theorem──► (theorem_str, difficulty)
                                            │
                                            ▼
CoderAgent.prove() ──LLM prompt──► Lean code
                                            │
                                    LeanTool.use()
                                            │
                           ┌────────────────┴────────────────┐
                           │ error?                           │ None
                           ▼                                  ▼
                  inject error_feedback             proof accepted ✓
                  into next LLM prompt
```

### Theoretical Grounding

> *"A machine could check any proof that a human could check, and moreover
> could discover proofs that are too long for any human to verify unaided."*
> — I.J. Good (1965)

References: de Moura et al. (2021) *Lean 4*; Avigad & Harrison (2014)
*Formally Verified Mathematics*; Good (1965).

Runtime: **~3 min** (mock mode, no Lean installation required; set `LEAN_PATH`
to enable real verification)

In [ ]:
# ── 0. Environment setup ────────────────────────────────────────────────────
import sys, os, importlib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
    print(f'Local mode — repo root: {repo_root}')

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── LeanTool (unchanged — still the canonical interface) ────────────────────
from prometheus.tools.base_tools import LeanTool, ProofState

# ── Production module imports (WP33) ────────────────────────────────────────
from prometheus.wp33_lean_theorem_proving import (
    TheoremDifficulty,
    TheoremCurriculum,
    ProofAttemptRecord,
    ProofSession,
    TheoremProver,
    TheoremProvingReport,
    verify_wp33_exit_criteria,
    _SEED_THEOREMS,
)

import prometheus
print(f'Prometheus version: {prometheus.__version__}')
print('WP33 production imports OK.')
print(f'  TheoremCurriculum : {TheoremCurriculum}')
print(f'  ProofSession      : {ProofSession}')
print(f'  TheoremProver     : {TheoremProver}')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 7), 'font.size': 11})

# Compatibility alias — used by visualisation cell
THEOREM_DIFFICULTIES = list(TheoremDifficulty.ALL)

In [ ]:
# ── 1. Configuration ────────────────────────────────────────────────────────
QUICK_MODE   = True
N_THEOREMS   = 6  if QUICK_MODE else 12   # theorems to attempt per difficulty
MAX_RETRIES  = 3                           # proof attempts per theorem

# LeanTool discovery:
#   use_lake=False forces bare `lean` binary (works with standalone .lean files).
#   use_lake=True  uses `lake`, which requires a full Lean project directory.
# For this notebook we always want the bare lean binary or the mock.
lean = LeanTool(use_lake=False, timeout=20)

mode_label = 'real Lean' if not lean._using_mock else 'sandboxed mock'
print(f'LeanTool mode:  {mode_label}')
if lean._using_mock:
    print('  → Install Lean 4 (https://leanprover.github.io) or set $LEAN_PATH')
    print('    to enable real compiler verification.')
else:
    print(f'  → Using binary: {lean._lean_exe}')
print()
print(f'Mode: {"QUICK" if QUICK_MODE else "FULL"}')
print(f'Theorems per difficulty: {N_THEOREMS} | Max retries: {MAX_RETRIES}')

# ── Build the production TheoremProver ──────────────────────────────────────
prover = TheoremProver(lean_tool=lean, n_per_level=N_THEOREMS, max_retries=MAX_RETRIES)
print()
print(f'TheoremProver ready (n_per_level={N_THEOREMS}, max_retries={MAX_RETRIES})')

---
## Section 1 — LeanTool Basics

We first demonstrate the core `LeanTool` API:
- `use(lean_code)` — verify a complete Lean 4 source file (returns `None` on success, error string on failure)
- `start_proof(theorem)` — initialise an interactive proof session
- `apply_tactic(proof_so_far, tactic)` — append a tactic and observe the new proof state

In [ ]:
# ── 2. LeanTool core API demonstration ──────────────────────────────────────

# 2a. Full-file verification
valid_lean = 'theorem add_zero (n : Nat) : n + 0 = n := by simp'
invalid_lean = 'theorem bad_proof (n : Nat) : n = n + 1 := by sorry'

err_valid   = lean.use(valid_lean)
err_invalid = lean.use(invalid_lean)

print('LeanTool.use() results:')
print(f'  Valid proof:   error={err_valid!r}   (None = verified OK)')
print(f'  Invalid proof: error={err_invalid!r}')
print()

# 2b. Interactive proof session
theorem = 'theorem add_comm_demo (a b : Nat) : a + b = b + a'
state, proof_so_far = lean.start_proof(theorem)
print(f'Initial state after start_proof():')
print(f'  {state.raw_state}')
print(f'  Complete: {state.is_complete()}')
print()

# Apply tactics step by step
for tactic in ['induction a with', 'case zero => simp', 'case succ n ih => ring']:
    state, proof_so_far = lean.apply_tactic(proof_so_far, tactic)
    print(f'  → tactic: `{tactic}`')
    print(f'    state:   {state.raw_state[:80]}')
    print(f'    complete:{state.is_complete()}')
print()
print(f'Using mock: {lean._using_mock}')

---
## Section 2 — CurriculumAgent Theorem Generation

`CurriculumAgent.generate_theorem()` now returns `(theorem_str, difficulty)`.
`generate_theorem_at_difficulty(difficulty)` lets us target a specific level directly.

In [ ]:
# ── 3. TheoremCurriculum — theorem generation ────────────────────────────────
# TheoremCurriculum replaces the hand-rolled poc_get_theorem helper.

curriculum = prover.curriculum   # reuse the one already inside the prover

print('Seed theorems by difficulty (from TheoremCurriculum / _SEED_THEOREMS):')
print()
for diff in TheoremDifficulty.ALL:
    seeds = _SEED_THEOREMS[diff]
    print(f'  [{diff.upper()}]')
    for s in seeds:
        print(f'    {s}')
    print()

print(f'TheoremCurriculum n_per_level = {curriculum.n_per_level}')
print(f'Theorem counts: {curriculum.summary()["theorem_counts"]}')

---
## Section 3 — Proof Attempt Loop

For each theorem, `CoderAgent.prove()` logic is demonstrated step-by-step:
1. Attempt to generate a proof via the LLM (or canonical proof in PoC mode).
2. Verify with `LeanTool.use()`.
3. On failure, inject the error message into the next attempt's prompt.
4. Accept the proof on success or give up after `MAX_RETRIES`.

In [ ]:
# ── 4. ProofSession — attempt→error→feedback→retry loop ─────────────────────
# ProofSession replaces poc_prove().  It wraps LeanTool with the same
# closed-loop feedback: attempt 0 (bad proof) → capture error → retry.

session = prover.session   # reuse from prover

# Smoke test: show the session at work on one theorem
thm = 'theorem add_zero (n : Nat) : n + 0 = n'
rec = session.prove(thm, TheoremDifficulty.EASY)

print(f'ProofSession smoke test:')
print(f'  Theorem : {thm}')
print(f'  Success : {rec.success}')
print(f'  Attempts: {rec.n_attempts}')
print(f'  Errors  : {rec.errors_seen}')
print(f'  Proof   : {rec.proof_text!r}')
print()
print('The error from attempt 0 is captured and would be injected into the')
print('next LLM prompt in a real (non-mock) deployment.')

In [ ]:
# ── 5. Main experiment — TheoremProver.prove_curriculum() ───────────────────
# One call replaces the nested loop over THEOREM_DIFFICULTIES.

report = prover.prove_curriculum()   # proves all three difficulty levels

# Build results_by_difficulty in the same shape as before for the viz cell
results_by_difficulty = {d: [] for d in THEOREM_DIFFICULTIES}
all_results = []

for rec in report.records:
    entry = {
        'theorem': rec.theorem,
        'proof': rec.proof_text,
        'attempts': rec.n_attempts,
        'errors': rec.errors_seen,
        'success': rec.success,
    }
    results_by_difficulty[rec.difficulty].append(entry)
    all_results.append({'difficulty': rec.difficulty, 'success': rec.success,
                        'attempts': rec.n_attempts})

# Print the per-theorem table
print(f'{"Diff":<8}  {"#":<3}  {"Theorem":<55}  {"Result":<7}  Attempts')
print('=' * 95)
for diff in THEOREM_DIFFICULTIES:
    for idx, r in enumerate(results_by_difficulty[diff]):
        status  = '✓ OK' if r['success'] else '✗ FAIL'
        preview = r['theorem'][:50] + '…' if len(r['theorem']) > 50 else r['theorem']
        print(f'{diff:<8}  {idx:<3}  {preview:<55}  {status:<7}  {r["attempts"]}')

print()
summ = prover.summary()
for diff in THEOREM_DIFFICULTIES:
    recs = results_by_difficulty[diff]
    n_ok = sum(r['success'] for r in recs)
    avg_att = np.mean([r['attempts'] for r in recs]) if recs else 0
    print(f'  [{diff.upper():6}]  {n_ok}/{len(recs)} proved  avg attempts: {avg_att:.1f}')

print()
print(f'Overall success rate: {report.overall_success_rate:.0%}  '
      f'({report.n_proved}/{report.n_theorems} proved)')

In [ ]:
# ── 6. Visualisation ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

diff_colors = {'easy': '#4CAF50', 'medium': '#FF9800', 'hard': '#F44336'}

# Panel A: Success rate per difficulty
ax = axes[0, 0]
diffs = THEOREM_DIFFICULTIES
success_rates = [
    sum(r['success'] for r in results_by_difficulty[d]) / max(len(results_by_difficulty[d]), 1)
    for d in diffs
]
bars = ax.bar(diffs, success_rates, color=[diff_colors[d] for d in diffs], edgecolor='black', alpha=0.85)
ax.bar_label(bars, fmt='%.0f%%', labels=[f'{r*100:.0f}%' for r in success_rates], fontsize=11)
ax.set_ylabel('Success Rate'); ax.set_ylim(0, 1.15)
ax.set_title('Proof Success Rate by Difficulty\n(% theorems proved within MAX_RETRIES)', fontweight='bold')

# Panel B: Average attempts per difficulty
ax2 = axes[0, 1]
avg_attempts = [
    np.mean([r['attempts'] for r in results_by_difficulty[d]])
    for d in diffs
]
bars2 = ax2.bar(diffs, avg_attempts, color=[diff_colors[d] for d in diffs], edgecolor='black', alpha=0.85)
ax2.bar_label(bars2, fmt='%.1f', fontsize=11)
ax2.set_ylabel('Average Attempts'); ax2.set_ylim(0, MAX_RETRIES + 0.5)
ax2.axhline(1.0, color='green', linestyle='--', alpha=0.5, label='1 attempt (ideal)')
ax2.set_title('Average Proof Attempts by Difficulty\n(1 = first-try success)', fontweight='bold')
ax2.legend()

# Panel C: Per-theorem attempt breakdown (scatter)
ax3 = axes[1, 0]
for diff in THEOREM_DIFFICULTIES:
    recs = results_by_difficulty[diff]
    x = [i + THEOREM_DIFFICULTIES.index(diff) * 0.25 - 0.25 for i in range(len(recs))]
    y = [r['attempts'] for r in recs]
    markers = ['o' if r['success'] else 'x' for r in recs]
    for xi, yi, mi in zip(x, y, markers):
        ax3.scatter(xi, yi, color=diff_colors[diff], marker=mi, s=80, zorder=3)

legend_elements = [
    mpatches.Patch(facecolor=diff_colors[d], label=d.capitalize()) for d in diffs
]
ax3.legend(handles=legend_elements, fontsize=9)
ax3.set_xlabel('Theorem index'); ax3.set_ylabel('Attempts')
ax3.set_title('Attempts per Theorem\n(circle=proved, x=failed)', fontweight='bold')
ax3.set_ylim(0, MAX_RETRIES + 0.5)
ax3.set_xticks(range(N_THEOREMS))
ax3.axhline(MAX_RETRIES, color='red', linestyle='--', alpha=0.4, label='Max retries')

# Panel D: LeanTool mode summary
ax4 = axes[1, 1]
total_proved = sum(r['success'] for r in all_results)
total = len(all_results)
labels = [f'Proved ({total_proved})', f'Failed ({total - total_proved})']
sizes  = [total_proved, total - total_proved]
colors = ['#4CAF50', '#F44336']
if total_proved == total:
    sizes = [total, 0.001]  # avoid zero-slice rendering issue
ax4.pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%',
        startangle=90, textprops={'fontsize': 11})
ax4.set_title(
    f'Overall Proof Outcome\n'
    f'(Mode: {mode_label}  |  {total_proved}/{total} proved)',
    fontweight='bold'
)

fig.suptitle(
    'WP33: Lean Theorem Proving — Prometheus v0.3\n'
    'LeanTool · CurriculumAgent · Iterative Proof with Error Feedback',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('wp33_lean_theorem_proving.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to wp33_lean_theorem_proving.png')

---
## Section 4 — Error Feedback Loop Inspection

The key insight of WP33 is that `CoderAgent.prove()` learns from its own
failures within a single theorem attempt.  We inspect the error messages
generated on the first (bad) attempt and show they are injected into
subsequent prompts.

In [ ]:
# ── 7. Error feedback loop inspection ───────────────────────────────────────
print('Error Feedback Loop Inspection')
print('=' * 60)
# Use the ProofAttemptRecord objects from the report directly
medium_records = [r for r in report.records if r.difficulty == TheoremDifficulty.MEDIUM]
sample_rec = medium_records[0]

print(f'Theorem : {sample_rec.theorem}')
print(f'Attempts: {sample_rec.n_attempts}')
print(f'Errors on attempt 0 (captured by ProofSession):')
for e in sample_rec.errors_seen:
    print(f'  └─ {e!r}')
print()
print(f'Final proof: {sample_rec.proof_text!r}')
print()
print('The sanitised error text is prepended as a Lean comment on the next')
print('attempt: "-- prev error: Proof contains [REDACTED]."')
print('This closed-loop feedback drives iterative proof search without')
print('contaminating the Lean source with the word "sorry".')
print()
# Show the full report dict
print('TheoremProvingReport summary:')
d = report.to_dict()
for k, v in d.items():
    if k != 'audit_log':
        print(f'  {k}: {v}')

In [ ]:
# ── 8. Exit-criteria verification — using production verify function ──────────
criteria = verify_wp33_exit_criteria(prover, report)

print('WP33 Exit Criteria Verification  (prometheus.wp33_lean_theorem_proving)')
print('=' * 65)
all_pass = True
for criterion, passed in criteria.items():
    status = '✓ PASS' if passed else '✗ FAIL'
    print(f'  {status}  {criterion}')
    if not passed:
        all_pass = False
print()
if all_pass:
    print('All WP33 exit criteria satisfied.')
    print(f'Lean theorem proving is operational (mode: {mode_label}).')
    print('TheoremCurriculum produces theorems; ProofSession verifies them;')
    print('error feedback drives iterative proof search.')
    print()
    print('Per-difficulty success rates:')
    for diff, rate in report.per_difficulty_rates.items():
        pct = f'{rate:.0%}' if rate is not None else 'n/a'
        print(f'  {diff:<8}: {pct}')
else:
    print('Some criteria not yet met.')

---
## Conclusions

**WP33 (v0.3 Task 3)** establishes the formal reasoning capability:

- **LeanTool** auto-discovers a real `lean`/`lake` binary when available,
  falls back to a sandboxed mock otherwise.  Error parsing handles both
  JSON diagnostic lines and plain-text output.
- **CurriculumAgent.generate_theorem()** returns `(theorem_str, difficulty)`
  at easy/medium/hard levels, seeding the proof curriculum.
- **CoderAgent.prove()** closes the loop: generate → verify → inject error
  → retry.  This is the same iterative error-feedback mechanism used for
  Python code but applied to a formal language.

### Foundation for v0.4 — The Autonomous Mathematician
WP33 is the prerequisite for v0.4's three tasks:
- **v0.4 T1**: Multi-step reasoning engine (`ProofState` management, `run_proof_cycle`)
- **v0.4 T2**: Proof tree search (`ProofTree`, backtracking, `PlannerAgent` as strategist)
- **v0.4 T3**: Meta-learning from failed proofs (`EvaluatorAgent.critique_failed_proof`)

### References
- de Moura, L. et al. (2021). *The Lean 4 Theorem Prover and Programming Language*. CADE.
- Avigad, J. & Harrison, J. (2014). *Formally Verified Mathematics*. CACM.
- Good, I.J. (1965). Speculations concerning the first ultraintelligent machine.
- Hofstadter, D. (1979). *Gödel, Escher, Bach*. Basic Books.